### Set up

In [1]:
# imports

from explore_import import  *
import tpp_preprocess as tpp
import hpp_checker as hpp
import data_preprocess as dt

import pyteomics.auxiliary as aux
from pathlib import Path
import os, re, subprocess
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
# base directories

root="/project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery"
data_dir=f"{root}/oui-discovery-vv-data/raw/tpp_pride_reanalysis/tpp_pride_reanalysis_arch230525"
processed_dir=f"{root}/oui-discovery-vv-data/processed/tpp_pride_reanalysis/tpp_pride_reanalysis_arch230525"
r_path="Rscript"
gwalk_work_dir=root
gwalk_script="Run_group_walk_tppfragpipe.R"

In [3]:
# replicate insides of data directory to preprocessed directory

#get insides of data directory
data_paths=dt.list_files(data_dir)

#replicate
for parent in data_paths.keys():
    new_parent=parent.replace(data_dir, processed_dir)
    os.makedirs(new_parent, exist_ok=True)

tpp_pride_reanalysis_arch230525/
    PXD014258/
        PXD014258-openprot/
            ESC-HF-Sample-MCF_openprot_PeptideProphet.pep.xml.index
            ESC-HF-Sample-MCF5.RAW.pep.xml
            ESC-HF-Sample-BT474_1_RAW.pep.xml
            ESC-HF-SampleHela_openprot_ProteinProphet.pep.prot.xml
            ESC-HF-SampleHela5.RAW.pep.xml
            ESC-HF-SampleHela_openprot_PeptideProphet.pep.xml.index
            ESC-HF-Sample-BT474_3_RAW.pep.xml
            ESC-HF-Sample-BT474_5_RAW.pep.xml
            ESC-HF-SampleHela3.RAW.pep.xml
            ESC-HF-Sample-MCF1.RAW.pep.xml
            ESC-HF-Sample-BT474_openprot_PeptideProphet.pep.xml
            ESC-HF-SampleHela1.RAW.pep.xml
            ESC-HF-Sample-MCF3.RAW.pep.xml
            ESC-HF-Sample-BT474_openprot_ProteinProphet.pep.prot.xml
            ESC-HF-SampleHela2.RAW.pep.xml
            ESC-HF-Sample-MCF_openprot_ProteinProphet.pep.prot.xml
            ESC-HF-Sample-MCF_openprot_PeptideProphet.pep.xml
            ESC-HF-S

### Load and process PeptideProphet files

In [4]:
def classify_leadprot(x):
    x=x.replace("decoy_","")
    if 'CONTAMINANT' in x.upper():
        return 'Contam'
    elif x.startswith('II_') or x.startswith('IP_'):
        return 'NonCanon'
        # Ensembl is canonical
    else:
        return 'Canon'

def is_peptide_canonical(x):
    '''x is the list of protein classes'''
    if np.array([_=='Contam' for _ in x]).any():
        return 'Contam'
    if np.array([_=='Canon' for _ in x]).any():
        return 'Canonical'
    return 'NonCanonical'

def classifiy_mods(row):
    if len(row.modifications)==0:
        return 'Unmodified'
    else:
        return 'Expected'

def custom_subgroup_filter(data_,key):
    filtered_subgroups = []
    for c,df in data_.groupby("FDRGroup").__iter__():
        tmp = aux.target_decoy.qvalues(df, key=key, reverse=True, is_decoy=df.database=='D',
                                      formula=1, full_output=True, q_label='custom_q')
        filtered_subgroups.append(tmp)

    return pd.concat(filtered_subgroups, ignore_index=True)

In [5]:
def col_isna(df,col):
    return 100-round((df[col].isna().value_counts()[False]/len(df))*100,4)

In [7]:
qval_score='peptideprophet_probability'

for parent, files in data_paths.items():
    new_parent=parent.replace(data_dir,processed_dir)
    dataset, database = parent.split("/")[-1].split("-")
    for file in files:
        if file.endswith("_PeptideProphet.pep.xml"):
            # read PeptideProphet output
            pepxml_path=os.path.join(parent, file)
            data=pepxml.DataFrame(pepxml_path)
            print(f"{dataset}|{database}|{file} N of discoveries: {len(data)}")
            
            #pin dataset and search_database and spectrum_file
            data["dataset"] = dataset
            data["search_database"] = database
            data["spectrum_file"] = file.replace("_PeptideProphet.pep.xml","").replace(f"_{database}","")
            
            #pin T/D database
            data["database"]=data["protein"].apply(tpp.get_database_tpp)
            td_counts = round((data["database"].value_counts()/len(data))*100,4)
            print(f"{dataset}|{database}|{file} targets and decoys and NA: {td_counts['T']} and {td_counts['D']} and {col_isna(data,'database')}")

            #calculate global q-value on peptideprophet_probability
            print(f"{dataset}|{database}|{file} {qval_score} range and NA: {min(data[qval_score])} - {max(data[qval_score])} and {col_isna(data,qval_score)}")
            data = aux.target_decoy.qvalues(data,
                                            key=qval_score,
                                            reverse=True,
                                            is_decoy=(data.database == 'D'),
                                            q_label='global_q',
                                            formula=1,
                                            full_output=True)
            print(f"{dataset}|{database}|{file} 'global_q' range and NA: {min(data['global_q'])} - {max(data['global_q'])} and {col_isna(data,'global_q')}")

            #pin helpfull labels
            data["peptide_class"]=data["protein"].apply(tpp.classify_peptide_tpp)
            
            #pin subgroups
            data['protein_classes'] = data.protein.apply(lambda x: np.unique([classify_leadprot(_) for _ in x]))
            data['isCanonical'] = data.protein_classes.apply(is_peptide_canonical)
            data['isModified']  = data.apply(classifiy_mods,axis=1)

            # prepare to group-wise
            data['isTarget'] = data.database.apply(lambda x: x=='T')
            #data['FDRGroup'] = data.isCanonical + '_' + data.isModified
            data['FDRGroup'] = data.isCanonical
            fdrgroup_counts = round((data["FDRGroup"].value_counts()/len(data))*100,4)
            counts_str = ", ".join(
                f"{key} {fdrgroup_counts[key]}%"
                for key in list(fdrgroup_counts.keys())
            )
            na_count = col_isna(data, 'FDRGroup')
            print(f"{dataset}|{database}|{file} {counts_str}, NA {na_count}")
            
            # calculate to group-wise
            data = custom_subgroup_filter(data, qval_score)
            print(f"{dataset}|{database}|{file} 'custom_q' range and NA: {min(data['custom_q'])} - {max(data['custom_q'])} and {col_isna(data,'custom_q')}")
            #add hybrid/group-wise
            #data["glob_cust_hybrid"]=data.apply(lambda x: x.custom_q if x.isCanonical=="NonCanonical" else x.global_q, axis=1)
            #print(f"{dataset}|{database}|{file} 'glob_cust_hybrid' range and NA: {min(data['glob_cust_hybrid'])} - {max(data['glob_cust_hybrid'])} and {col_isna(data,'glob_cust_hybrid')}")
        
            
            #save for Group-walk
            basename = os.path.basename(pepxml_path).split('.')[0]
            gwalk_input = os.path.join(new_parent, f"{basename}.csv")
            data.to_csv(gwalk_input)
            
            #run Group-walk
            dataset_dir = new_parent
            working_dir = gwalk_work_dir
            file_name = os.path.basename(gwalk_input)
            print(f"Run Group-walk on {gwalk_input}")
            command = (
                f"module load r && {r_path} {gwalk_script} {dataset_dir} {working_dir} {file_name}"
            )
            
            _ = subprocess.run(command, shell=True, check=True)
            
            qwalk_output = os.path.join(new_parent,"groupwalk_output_"+file_name)
            
            #check group-walk
            data2=pd.read_csv(qwalk_output)
            #data2['glob_group_hybrid']=data2.apply(lambda x: x.group_q_prob if x.isCanonical=="NonCanonical" else x.global_q, axis=1)
            print(f"{dataset}|{database}|{file} 'group_q_prob' range and NA: {min(data2['group_q_prob'])} - {max(data2['group_q_prob'])} and {col_isna(data2,'group_q_prob')}")
            #data2.to_csv(qwalk_output)
            del data2
            
#            break
#    break

PXD014258|openprot|ESC-HF-Sample-BT474_openprot_PeptideProphet.pep.xml N of discoveries: 48292
PXD014258|openprot|ESC-HF-Sample-BT474_openprot_PeptideProphet.pep.xml targets and decoys and NA: 94.1771 and 5.8229 and 0.0
PXD014258|openprot|ESC-HF-Sample-BT474_openprot_PeptideProphet.pep.xml peptideprophet_probability range and NA: 0.05 - 1.0 and 0.0
PXD014258|openprot|ESC-HF-Sample-BT474_openprot_PeptideProphet.pep.xml 'global_q' range and NA: 0.0 - 0.06182937554969217 and 0.0
PXD014258|openprot|ESC-HF-Sample-BT474_openprot_PeptideProphet.pep.xml Canonical 84.4053%, Contam 8.5356%, NonCanonical 7.0591%, NA 0.0
PXD014258|openprot|ESC-HF-Sample-BT474_openprot_PeptideProphet.pep.xml 'custom_q' range and NA: 0.0 - 0.809447983014862 and 0.0
Run Group-walk on /project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery/oui-discovery-vv-data/processed/tpp_pride_reanalysis/tpp_pride_reanalysis_arch230525/PXD014258/PXD014258-openprot/ESC-HF-Sample-BT474_openprot_PeptideProphet.csv
PXD0

### PSM -> peptide

In [8]:
processed_paths=dt.list_files(processed_dir)

tpp_pride_reanalysis_arch230525/
    PXD014258/
        PXD014258-openprot/
            ESC-HF-SampleHela_openprot_PeptideProphet.csv
            groupwalk_output_ESC-HF-SampleHela_openprot_PeptideProphet.csv
            groupwalk_output_ESC-HF-Sample-BT474_openprot_PeptideProphet.csv
            ESC-HF-Sample-MCF_openprot_PeptideProphet.csv
            groupwalk_output_ESC-HF-Sample-MCF_openprot_PeptideProphet.csv
            ESC-HF-Sample-BT474_openprot_PeptideProphet.csv
            .ipynb_checkpoints/
                groupwalk_output_ESC-HF-Sample-BT474_openprot_PeptideProphet-checkpoint.csv
        PXD014258-trembl/
            ESC-HF-SampleHela_trembl_PeptideProphet.csv
            groupwalk_output_ESC-HF-Sample-MCF_trembl_PeptideProphet.csv
            groupwalk_output_ESC-HF-SampleHela_trembl_PeptideProphet.csv
            groupwalk_output_ESC-HF-Sample-BT474_trembl_PeptideProphet.csv
            ESC-HF-Sample-MCF_trembl_PeptideProphet.csv
            ESC-HF-Sample-BT474_trembl

In [14]:
qval_score='peptideprophet_probability'

for parent, files in processed_paths.items():
    if ".ipynb_checkpoints" in parent: continue
    database, dataset = parent.split("/")[-1].split("-")
    for file in files:
        # omit group-walk output PSM-level and newly added peptide-level and group-walk output peptide-level
        if (not file.startswith("groupwalk_output_")) and (not file.endswith("_pep.csv")) :
            spectrum_file = file.split(".")[0]

            data = pd.read_csv(os.path.join(parent, file))
            print(f"PSM -> peptide : {file}")
            data.drop(['global_q','custom_q'],inplace=True,axis=1)

            # select high-scoring PSMs
            data_peptide = data.sort_values(qval_score, ascending=False).drop_duplicates("peptide", keep="first")
            print(f"Number of PSMs and peptides: {len(data)} , {len(data_peptide)}")
            del data

            #pin T/D database
            td_counts = round((data_peptide["database"].value_counts()/len(data_peptide))*100,4)
            print(f"{dataset}|{database}|{file} targets and decoys and NA: {td_counts['T']} and {td_counts['D']} and {col_isna(data_peptide,'database')}")

            #calculate global q-value
            print(f"{dataset}|{database}|{file} {qval_score} range and NA: {min(data_peptide[qval_score])} - {max(data_peptide[qval_score])} and {col_isna(data_peptide,qval_score)}")
            data_peptide = aux.target_decoy.qvalues(data_peptide,
                                            key=qval_score,
                                            reverse=True,
                                            is_decoy=(data_peptide.database == 'D'),
                                            q_label='global_q',
                                            formula=1,
                                            full_output=True)
            print(f"{dataset}|{database}|{file} 'global_q' range and NA: {min(data_peptide['global_q'])} - {max(data_peptide['global_q'])} and {col_isna(data_peptide,'global_q')}")

            # calculate group-wise q-value
            fdrgroup_counts = round((data_peptide["FDRGroup"].value_counts()/len(data_peptide))*100,4)
            counts_str = ", ".join(
                f"{key} {fdrgroup_counts[key]}%"
                for key in list(fdrgroup_counts.keys())
            )
            na_count = col_isna(data_peptide, 'FDRGroup')
            print(f"{dataset}|{database}|{file} {counts_str}, NA {na_count}")
            data_peptide = custom_subgroup_filter(data_peptide, qval_score)
            print(f"{dataset}|{database}|{file} 'custom_q' range and NA: {min(data_peptide['custom_q'])} - {max(data_peptide['custom_q'])} and {col_isna(data_peptide,'custom_q')}")

            #save for Group-walk
            gwalk_input = os.path.join(parent, f"{spectrum_file}_pep.csv")
            data_peptide.to_csv(gwalk_input,index=False)

            #run Group-walk
            dataset_dir = parent
            working_dir = gwalk_work_dir
            file_name = os.path.basename(gwalk_input)
            print(f"Run Group-walk on {gwalk_input}")
            command = (
                f"module load r && {r_path} {gwalk_script} {dataset_dir} {working_dir} {file_name}"
            )
            
            _ = subprocess.run(command, shell=True, check=True)
            
            qwalk_output = os.path.join(parent,"groupwalk_output_"+file_name)

            #check group-walk
            data_peptide2=pd.read_csv(qwalk_output)
            print(f"{dataset}|{database}|{file} 'group_q_prob' range and NA: {min(data_peptide2['group_q_prob'])} - {max(data_peptide2['group_q_prob'])} and {col_isna(data_peptide2,'group_q_prob')}")
            del data_peptide2
            
#            break
#        break

PSM -> peptide : ESC-HF-SampleHela_openprot_PeptideProphet.csv
Number of PSMs and peptides: 56726 , 34975
openprot|PXD014258|ESC-HF-SampleHela_openprot_PeptideProphet.csv targets and decoys and NA: 76.9492 and 23.0508 and 0.0
openprot|PXD014258|ESC-HF-SampleHela_openprot_PeptideProphet.csv peptideprophet_probability range and NA: 0.05 - 1.0 and 0.0
openprot|PXD014258|ESC-HF-SampleHela_openprot_PeptideProphet.csv 'global_q' range and NA: 0.00019657951641438963 - 0.29955783450377144 and 0.0
openprot|PXD014258|ESC-HF-SampleHela_openprot_PeptideProphet.csv Canonical 70.6705%, NonCanonical 27.2252%, Contam 2.1044%, NA 0.0
openprot|PXD014258|ESC-HF-SampleHela_openprot_PeptideProphet.csv 'custom_q' range and NA: 0.0 - 0.9155099577549789 and 0.0
Run Group-walk on /project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery/oui-discovery-vv-data/processed/tpp_pride_reanalysis/tpp_pride_reanalysis_arch230525/PXD005833/PXD005833-canon/ESC-HF-SampleHela_openprot_PeptideProphet_pep.csv
op